# VayuSense — GPU Acceleration Benchmark (cuDF vs pandas)

**Run on Google Colab with a T4 GPU runtime** (`Runtime → Change runtime type → T4 GPU`).

This notebook:
1. Discovers Delhi-NCR air-quality stations via the OpenAQ API
2. Bulk-downloads multi-year raw sensor readings from the public OpenAQ S3 archive (AWS Open Data)
3. Runs the identical analytics pipeline (clean → aggregate → rolling trends → anomalies) on **CPU (pandas)** and **GPU (NVIDIA cuDF/RAPIDS)**
4. Produces the benchmark chart + processed datasets used by the VayuSense app

> Set your free OpenAQ API key below (sign up at https://explore.openaq.org — Account → API key).

In [ ]:
OPENAQ_API_KEY = ""  # <-- paste your key here

# Delhi-NCR bounding search + extra APAC cities for scale
CITY_QUERIES = {
    "Delhi":   {"coordinates": "28.6139,77.2090", "radius": 25000},
    "Mumbai":  {"coordinates": "19.0760,72.8777", "radius": 25000},
    "Kolkata": {"coordinates": "22.5726,88.3639", "radius": 25000},
    "Chennai": {"coordinates": "13.0827,80.2707", "radius": 25000},
}
YEARS = [2024, 2025]           # archive years to pull
MAX_LOCATIONS_PER_CITY = 12    # cap stations per city
PARAMS = {"pm25", "pm10", "no2", "o3", "so2", "co"}
print("config ok")

In [ ]:
# 1) Discover station location IDs via OpenAQ v3 API
import requests

HEADERS = {"X-API-Key": OPENAQ_API_KEY}
locations = {}  # location_id -> (city, name)
for city, q in CITY_QUERIES.items():
    r = requests.get(
        "https://api.openaq.org/v3/locations",
        params={"coordinates": q["coordinates"], "radius": q["radius"], "limit": 100},
        headers=HEADERS, timeout=30)
    r.raise_for_status()
    results = r.json()["results"]
    # prefer stations with most sensors (richer data)
    results.sort(key=lambda x: len(x.get("sensors", [])), reverse=True)
    for loc in results[:MAX_LOCATIONS_PER_CITY]:
        locations[loc["id"]] = (city, loc["name"])
print(f"Discovered {len(locations)} stations across {len(CITY_QUERIES)} cities")
list(locations.items())[:5]

In [ ]:
# 2) Bulk-download raw daily CSVs from the public OpenAQ S3 archive (no AWS account needed)
import concurrent.futures as cf, io, gzip, xml.etree.ElementTree as ET
import pandas as pd

S3 = "https://openaq-data-archive.s3.amazonaws.com"
NS = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}

def list_keys(prefix):
    keys, token = [], None
    while True:
        params = {"list-type": "2", "prefix": prefix, "max-keys": "1000"}
        if token: params["continuation-token"] = token
        r = requests.get(S3, params=params, timeout=30)
        root = ET.fromstring(r.text)
        keys += [c.find("s3:Key", NS).text for c in root.findall("s3:Contents", NS)]
        token_el = root.find("s3:NextContinuationToken", NS)
        if token_el is None: break
        token = token_el.text
    return keys

def fetch_csv(key):
    try:
        r = requests.get(f"{S3}/{key}", timeout=60)
        if r.status_code != 200: return None
        return pd.read_csv(io.BytesIO(gzip.decompress(r.content)))
    except Exception:
        return None

all_keys = []
for lid in locations:
    for y in YEARS:
        all_keys += list_keys(f"records/csv.gz/locationid={lid}/year={y}/")
print(f"{len(all_keys)} daily archive files to download...")

frames = []
with cf.ThreadPoolExecutor(max_workers=32) as ex:
    for i, df in enumerate(ex.map(fetch_csv, all_keys)):
        if df is not None: frames.append(df)
        if (i+1) % 500 == 0: print(f"  {i+1}/{len(all_keys)}")

raw = pd.concat(frames, ignore_index=True)
raw = raw[raw["parameter"].isin(PARAMS)]
city_map = {lid: c for lid, (c, _) in locations.items()}
raw["city"] = raw["location_id"].map(city_map)
print(f"RAW DATASET: {len(raw):,} sensor readings")
raw.head()

In [ ]:
# 3) The analytics pipeline — IDENTICAL code path for pandas (CPU) and cuDF (GPU)
import time

def run_pipeline(df, lib):
    """clean -> hourly agg -> daily agg -> rolling trends -> anomaly flags -> station league"""
    t0 = time.perf_counter()
    d = df.copy()
    # clean
    d = d[(d["value"] >= 0) & (d["value"] < 2000)]
    d["ts"] = lib.to_datetime(d["datetime"], errors="coerce", utc=True)
    d = d.dropna(subset=["ts"])
    d["hour"] = d["ts"].dt.floor("h")
    d["date"] = d["ts"].dt.floor("D")
    # hourly per city/parameter
    hourly = d.groupby(["city", "parameter", "hour"], as_index=False)["value"].mean()
    # daily per city/parameter
    daily = d.groupby(["city", "parameter", "date"], as_index=False)["value"].agg(["mean", "max", "count"]).reset_index()
    # rolling 7-day trend on daily means (per city+param)
    daily = daily.sort_values(["city", "parameter", "date"])
    daily["roll7"] = daily.groupby(["city", "parameter"])["mean"].rolling(7, min_periods=1).mean().reset_index(drop=True)
    # anomaly flags: z-score of daily mean within city+param
    stats = daily.groupby(["city", "parameter"])["mean"].agg(["mean", "std"]).reset_index()
    stats.columns = ["city", "parameter", "mu", "sigma"]
    daily = daily.merge(stats, on=["city", "parameter"])
    daily["zscore"] = (daily["mean"] - daily["mu"]) / daily["sigma"]
    daily["anomaly"] = daily["zscore"].abs() > 2.0
    # station league: worst PM2.5 stations
    pm = d[d["parameter"] == "pm25"]
    league = pm.groupby(["city", "location"], as_index=False)["value"].mean().sort_values("value", ascending=False)
    elapsed = time.perf_counter() - t0
    return hourly, daily, league, elapsed

# CPU run (pandas)
hourly_pd, daily_pd, league_pd, t_cpu = run_pipeline(raw, pd)
print(f"CPU (pandas): {t_cpu:.2f}s over {len(raw):,} rows")

In [ ]:
# GPU run (NVIDIA cuDF) — same pipeline, same data
import cudf
graw = cudf.from_pandas(raw)
# warmup (JIT/alloc)
_ = run_pipeline(graw, cudf)
hourly_gpu, daily_gpu, league_gpu, t_gpu = run_pipeline(graw, cudf)
print(f"GPU (cuDF):   {t_gpu:.2f}s over {len(raw):,} rows")
print(f"SPEEDUP: {t_cpu/t_gpu:.1f}x faster on NVIDIA T4 GPU")

In [ ]:
# 4) Benchmark chart + results artifacts
import json, matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.2))
bars = ax.bar(["pandas (CPU)", "cuDF (NVIDIA T4 GPU)"], [t_cpu, t_gpu], color=["#9aa0a6", "#76b900"])
ax.bar_label(bars, fmt="%.2f s", fontsize=12, fontweight="bold")
ax.set_ylabel("Pipeline time (seconds)")
ax.set_title(f"VayuSense analytics pipeline — {len(raw):,} sensor readings\n{t_cpu/t_gpu:.1f}× faster with NVIDIA cuDF", fontsize=12)
plt.tight_layout(); plt.savefig("benchmark.png", dpi=160)
plt.show()

with open("benchmark_results.json", "w") as f:
    json.dump({"rows": int(len(raw)), "cpu_seconds": round(t_cpu, 3),
               "gpu_seconds": round(t_gpu, 3), "speedup": round(t_cpu/t_gpu, 2),
               "gpu": "NVIDIA T4 (Google Colab)", "lib": "cuDF / RAPIDS"}, f, indent=2)
print("saved benchmark.png + benchmark_results.json")

In [ ]:
# 5) Export processed datasets for the VayuSense app (small, committable)
hourly_out = hourly_gpu.to_pandas() if hasattr(hourly_gpu, "to_pandas") else hourly_pd
daily_out = daily_gpu.to_pandas() if hasattr(daily_gpu, "to_pandas") else daily_pd
league_out = league_gpu.to_pandas() if hasattr(league_gpu, "to_pandas") else league_pd

daily_out.to_parquet("daily_city.parquet", index=False)
hourly_out.tail(24*90*len(CITY_QUERIES)*6).to_parquet("hourly_recent.parquet", index=False)
league_out.to_parquet("station_league.parquet", index=False)

from google.colab import files
for f in ["benchmark.png", "benchmark_results.json", "daily_city.parquet", "hourly_recent.parquet", "station_league.parquet"]:
    files.download(f)
print("artifacts downloaded — put them in vayusense/benchmark/ and vayusense/data/processed/")